# Penerapan Metode IndoBERT untuk Mengukur Kemiripan Berdasarkan Judul Konten pada Sistem Rekomendasi Channel YouTube

**Alur Pipeline:**
1. Install & Import Library
2. Load Dataset JSON 
3. Pengecekan Missing Value
4. Pengecekan Duplikat
5. Text Cleaning
6. Case Folding
7. Tokenisasi dengan Stanza (Bahasa Indonesia)
8. Split Data Terintegrasi 70/10/20 (Train/Validation/Test Channel)
9. Konfigurasi Model IndoBERT
10. Fine-Tuning IndoBERT (Supervised Classification)
11. Generate Embedding Judul Video (Fine-Tuned Encoder)
12. Agregasi Vektor Channel (Mean Pooling + L2 Normalization)
13. Cosine Similarity Matrix & Fungsi Rekomendasi
14. Evaluasi Sistem (Precision@K pada Test Channels)
15. Simpan Artefak Final Fine-Tuning

---
## 1. Install & Import Library

In [1]:
# Install library yang diperlukan
%pip install torch transformers stanza scikit-learn pandas numpy tqdm --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Import library inti dan siapkan helper untuk deteksi device
import json
import re
import os
import numpy as np
import pandas as pd

# Deep Learning & NLP
import torch
from transformers import AutoTokenizer

# Stanza untuk Tokenisasi Bahasa Indonesia
import stanza

# Similarity & Evaluation
from sklearn.metrics.pairwise import cosine_similarity

# Progress bar
from tqdm import tqdm

# Konfigurasi device (GPU jika tersedia, fallback ke CPU jika tidak kompatibel)
def get_device():
    """Deteksi device dengan fallback ke CPU jika CUDA tidak kompatibel."""
    if not torch.cuda.is_available():
        return torch.device('cpu')

    try:
        major, minor = torch.cuda.get_device_capability(0)
        # PyTorch build yang terpasang di environment ini hanya mendukung CC >= 7.5
        if (major, minor) < (7, 5):
            print(
                f"⚠️  GPU {torch.cuda.get_device_name(0)} memiliki compute capability "
                f"sm_{major}{minor}, jadi fallback ke CPU"
            )
            return torch.device('cpu')

        # Test ringan agar benar-benar memastikan CUDA bisa dipakai
        _ = torch.randn(1, device='cuda')
        return torch.device('cuda')
    except Exception as e:
        print(f"⚠️  CUDA tidak bisa dipakai ({e}) → fallback ke CPU")
        return torch.device('cpu')

device = get_device()
print(f'Menggunakan device  : {device}')
print(f'PyTorch version     : {torch.__version__}')
print('Import library selesai.')

/home/candimadam/Documents/Tugas Akhir/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⚠️  GPU NVIDIA GeForce MX350 memiliki compute capability sm_61, jadi fallback ke CPU
Menggunakan device  : cpu
PyTorch version     : 2.11.0+cu130
Import library selesai.


/home/candimadam/Documents/Tugas Akhir/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU0 NVIDIA GeForce MX350 which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/candimadam/Documents/Tugas Akhir/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:489: UserWarning: 
NVIDIA GeForce MX350 with CUDA capability sm_61 is not compatible with the curre

---
## 2. Load Dataset JSON

In [ ]:
# Load dataset JSON dan ubah ke DataFrame
DATA_PATH = 'data_video.json'

# Load JSON
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Konversi ke DataFrame
df = pd.DataFrame(raw_data)

# Informasi Dataset
print('=' * 55)
print('           INFORMASI DATASET')
print('=' * 55)
print(f'Total record (video)   : {len(df):,}')
print(f'Total kolom            : {len(df.columns)}')
print(f'Nama kolom             : {list(df.columns)}')
print(f'Jumlah channel unik    : {df["nama_channel"].nunique()}')
print(f'Jumlah kategori unik   : {df["kategori"].nunique()}')
print('=' * 55)

# Distribusi video per channel
dist = df['nama_channel'].value_counts()
print(f'\nDistribusi jumlah video per channel:')
print(f'  Min  : {dist.min()} video')
print(f'  Max  : {dist.max()} video')
print(f'  Rata : {dist.mean():.1f} video')
print()

# Tampilkan 5 baris pertama
df.head()

           INFORMASI DATASET
Total record (video)   : 10,000
Total kolom            : 9
Nama kolom             : ['id', 'link_channel', 'nama_channel', 'kategori', 'jumlah_pelanggan', 'judul', 'link', 'jumlah_tayangan', 'tanggal_upload']
Jumlah channel unik    : 100
Jumlah kategori unik   : 10

Distribusi jumlah video per channel:
  Min  : 100 video
  Max  : 100 video
  Rata : 100.0 video



,id,link_channel,nama_channel,kategori,jumlah_pelanggan,judul,link,jumlah_tayangan,tanggal_upload
0,1,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP...,https://www.youtube.com/watch?v=zE5H9KQ_Hyg,1700000,5 days ago
1,2,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,RAJA TERAKHIR HP SAMSUNG!,https://www.youtube.com/watch?v=snB4jbtscxU,932000,10 days ago
2,3,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...,https://www.youtube.com/watch?v=3KBJtbEAdzs,802000,11 days ago
3,4,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,"Kalau Apple niat, iPhone bisa seworth it ini.....",https://www.youtube.com/watch?v=rkLpVyRGCPw,1300000,2 weeks ago
4,5,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,Xiaomi pun ngeluh soal fenomena ini...,https://www.youtube.com/watch?v=Z4m_fwJ5eHQ&pp...,1200000,2 weeks ago


---
## 3. Pengecekan Missing Value

Sebelum preprocessing teks, notebook melakukan pengecekan kualitas data dasar:
- **Missing value**: melihat kolom mana yang kosong


In [ ]:
# Hitung jumlah missing value per kolom dan urutkan dari yang terbanyak
missing_per_column = df.isna().sum().sort_values(ascending=False)

# Total sel yang kosong di seluruh DataFrame
missing_total = int(missing_per_column.sum())

# Jumlah baris yang memiliki setidaknya satu missing value
rows_with_missing = int(df.isna().any(axis=1).sum())

# Tampilan ringkas hasil pengecekan
print('=' * 70)
print('PENGECEKAN MISSING VALUE')
print('=' * 70)
print(f'Total sel missing value      : {missing_total}')
print(f'Jumlah baris yang memiliki missing value: {rows_with_missing}')
print('\nMissing value per kolom:')
# .to_string() agar tampil lengkap tanpa terpotong ketika Series panjang
print(missing_per_column.to_string())

# Jika ada missing value, tampilkan beberapa contoh baris yang terpengaruh
if missing_total > 0:
    print('\nContoh baris yang memiliki missing value:')
    display(df[df.isna().any(axis=1)].head(5))
else:
    print('\nTidak ditemukan missing value pada dataset.')

PENGECEKAN MISSING VALUE
Total sel missing value      : 0
Jumlah baris yang memiliki missing value: 0

Missing value per kolom:
id                  0
link_channel        0
nama_channel        0
kategori            0
jumlah_pelanggan    0
judul               0
link                0
jumlah_tayangan     0
tanggal_upload      0

Tidak ditemukan missing value pada dataset.


---
## 4. Pengecekan Duplikat

Sebelum preprocessing teks, notebook melakukan pengecekan kualitas data dasar:
- **Duplikat**: melihat data video yang terduplikasi


In [ ]:
# Cari baris yang duplikat identik agar data lebih bersih
exact_duplicate_count = int(df.duplicated().sum())

# Ambil semua baris yang terduplikasi 
duplicate_rows = df[df.duplicated(keep=False)].copy()

print('=' * 70)
print('PENGECEKAN DUPLIKAT')
print('=' * 70)
# Tampilkan jumlah total baris yang duplikat identik
print(f'Jumlah baris duplikat identik : {exact_duplicate_count}')

# Jika ada duplikat, tampilkan contoh baris-baris yang duplikat
if exact_duplicate_count > 0:
    print('\nContoh baris yang duplikat:')
    display(duplicate_rows.head(10))
else:
    print('\nTidak ditemukan baris duplikat identik pada dataset.')

PENGECEKAN DUPLIKAT
Jumlah baris duplikat identik : 0

Tidak ditemukan baris duplikat identik pada dataset.


---
## 5. Text Cleaning

Pembersihan teks secara **minimalis**: hanya menghapus noise tanpa makna bahasa
(emoji, simbol dekoratif non-standar, karakter encoding rusak).
Tanda baca standar **(titik, koma, tanda tanya)** tetap dipertahankan karena IndoBERT memanfaatkannya.

    Tahapan:
    1. Menghapus emoji dan simbol Unicode non-standar
    2. Menghapus karakter encoding yang rusak / tidak dikenali
    3. Menghapus karakter khusus / dekoratif yang bukan tanda baca standar
    4. Merapikan spasi berlebih

In [ ]:
# Bersihkan judul video dari noise non-linguistik tanpa menghapus tanda baca penting
def text_cleaning(text):
    if not isinstance(text, str):
        return ''

    # 1. Hapus emoji dan simbol Unicode non-standar
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"   # emoticons wajah (😀 😁 😂 😍 😭 😡 🙏)
        "\U0001F300-\U0001F5FF"   # simbol & piktogram (🌟 🌈 🌍 🌙 🔥 💡 🎉)
        "\U0001F680-\U0001F6FF"   # transport & peta (🚗 🚕 🚌 🚆 ✈️ 🚀 🚢)
        "\U0001F1E0-\U0001F1FF"   # bendera (🇮🇩 🇺🇸 🇯🇵 🇰🇷 🇬🇧)
        "\U00002600-\U000026FF"   # simbol umum (☀️ ☁️ ☂️ ☕ ☎️ ♨️ ⚽)
        "\U00002700-\U000027BF"   # Dingbats (✂️ ✈️ ✉️ ✅ ❌ ❗ ❤)
        "\U0001F900-\U0001F9FF"   # simbol tambahan (🤣 🤖 🤩 🤯 🤔 🥳 🧠 🦾)
        "\U0001FA00-\U0001FA6F"   # simbol tambahan-A (🩰 🩸 🪀 🪁 🪐 🪑)
        "\U0001FA70-\U0001FAFF"   # simbol tambahan-B (🩷 🩵 🩶 🪪 🪫 🫠 🫶)
        "\U00002300-\U000023FF"   # teknis (⌚ ⌛ ⏰ ⏱️ ⏲️ ⏳)
        "]+",
        flags=re.UNICODE
    )
    text = emoji_pattern.sub(' ', text)

    # 2. Hapus karakter non-printable / encoding rusak
    text = re.sub(r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F]', ' ', text)

    # 3. Hapus simbol dekoratif yang bukan tanda baca standar
    #    Pertahankan: huruf, angka, spasi, . , ? ! - ( ) / @ # % + = : ; ' "
    text = re.sub(r'[^\w\s.,?!\-()/@#%+=\'":;]', ' ', text, flags=re.UNICODE)

    # 4. Rapikan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Terapkan Text Cleaning
df['judul_clean'] = df['judul'].apply(text_cleaning)

# Tampilkan contoh hasil cleaning
print('Contoh Hasil Text Cleaning:')
print('=' * 70)
for _, row in df[['judul', 'judul_clean']].head(8).iterrows():
    print(f'SEBELUM : {row["judul"]}')
    print(f'SESUDAH : {row["judul_clean"]}')
    print('-' * 70)

Contoh Hasil Text Cleaning:
SEBELUM : Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP ASLINYA...
SESUDAH : Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP ASLINYA...
----------------------------------------------------------------------
SEBELUM : RAJA TERAKHIR HP SAMSUNG!
SESUDAH : RAJA TERAKHIR HP SAMSUNG!
----------------------------------------------------------------------
SEBELUM : Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...
SESUDAH : Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...
----------------------------------------------------------------------
SEBELUM : Kalau Apple niat, iPhone bisa seworth it ini... - Review iPhone 17
SESUDAH : Kalau Apple niat, iPhone bisa seworth it ini... - Review iPhone 17
----------------------------------------------------------------------
SEBELUM : Xiaomi pun ngeluh soal fenomena ini...
SESUDAH : Xiaomi pun ngeluh soal fenomena ini...
----------------------------------------------------------------------
SEBELUM : Rekomendasi HP TERBAIK buat A

---
## 6. Case Folding

Mengubah seluruh teks menjadi huruf kecil (**lowercase**) agar sesuai dengan
model **IndoBERT Base Uncased** yang dilatih pada teks huruf kecil.

In [ ]:
# Ubah seluruh judul menjadi huruf kecil agar konsisten dengan model uncased
def case_folding(text):
    # Jika input bukan string, kembalikan string kosong untuk menghindari error
    if not isinstance(text, str):
        return ''
    # Ubah semua karakter menjadi lowercase (sesuai kebutuhan model uncased)
    return text.lower()

# Terapkan case folding setelah text cleaning
# Simpan ke kolom baru 'judul_lower' agar kolom original tetap tersedia untuk debug/analisis
df['judul_lower'] = df['judul_clean'].apply(case_folding)

# Tampilkan contoh hasil untuk verifikasi
print('Contoh Hasil Case Folding (setelah Text Cleaning):')
print('=' * 70)
# Cetak beberapa baris pertama (clean -> lower) untuk memastikan transformasi berjalan benar
for _, row in df[['judul_clean', 'judul_lower']].head(5).iterrows():
    print(f'SEBELUM : {row["judul_clean"]}')
    print(f'SESUDAH : {row["judul_lower"]}')
    print('-' * 70)

Contoh Hasil Case Folding (setelah Text Cleaning):
SEBELUM : Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP ASLINYA...
SESUDAH : unboxing iphone 17 pro palsu yang sangat mirip aslinya...
----------------------------------------------------------------------
SEBELUM : RAJA TERAKHIR HP SAMSUNG!
SESUDAH : raja terakhir hp samsung!
----------------------------------------------------------------------
SEBELUM : Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...
SESUDAH : rp1.599 juta! ketika oppo niat bikin hp murah...
----------------------------------------------------------------------
SEBELUM : Kalau Apple niat, iPhone bisa seworth it ini... - Review iPhone 17
SESUDAH : kalau apple niat, iphone bisa seworth it ini... - review iphone 17
----------------------------------------------------------------------
SEBELUM : Xiaomi pun ngeluh soal fenomena ini...
SESUDAH : xiaomi pun ngeluh soal fenomena ini...
----------------------------------------------------------------------


---
## 7. Tokenisasi dengan Stanza (Bahasa Indonesia)

Stanza digunakan untuk **tokenisasi** teks Bahasa Indonesia.
Output berupa teks yang sudah dinormalisasi (token yang bergabung kembali).

In [ ]:
# Download model Bahasa Indonesia
stanza.download('id', verbose=False)   # 'id' = kode bahasa Indonesia
print('Model Stanza Bahasa Indonesia siap.')

Model Stanza Bahasa Indonesia siap.


In [ ]:
# Siapkan pipeline Stanza untuk tokenisasi bahasa Indonesia
# Hanya processor 'tokenize' yang diaktifkan untuk efisiensi
try:
    # Coba inisialisasi Stanza dengan GPU (lebih cepat)
    nlp_stanza = stanza.Pipeline(
        lang='id',  # 'id' = kode bahasa Indonesia
        processors='tokenize',  # Hanya tokenize, skip POS tagging, lemmatization, dll
        verbose=False
    )
    print("✓ Stanza initialized on GPU")
except RuntimeError as e:
    # Jika GPU tidak kompatibel, gunakan CPU sebagai fallback
    print(f"⚠️  GPU error: {str(e)[:80]}...")
    print("   Fallback ke CPU...")
    nlp_stanza = stanza.Pipeline(
        lang='id',
        processors='tokenize',
        device='cpu',  # Paksa gunakan CPU
        verbose=False
    )
    print("✓ Stanza initialized on CPU")

# Fungsi untuk memecah teks menjadi token individual menggunakan Stanza.
def tokenize_stanza(text):
    
    # Validasi input: jika text kosong atau hanya spasi, kembalikan apa adanya
    if not text or not text.strip():
        return text

    # Proses teks dengan Stanza untuk tokenisasi
    doc = nlp_stanza(text)
    
    # Kumpulkan semua token dari semua kalimat yang dideteksi Stanza
    tokens = []
    for sentence in doc.sentences:  # Iterasi setiap kalimat
        for token in sentence.tokens:  # Iterasi setiap token dalam kalimat
            tokens.append(token.text)  # Ambil teks token

    # Gabungkan semua token dengan spasi untuk menghasilkan teks tertokenisasi
    return ' '.join(tokens)

# Terapkan tokenisasi ke seluruh dataset
print(f'Memproses tokenisasi Stanza untuk {len(df):,} judul...')
print('(Proses ini memerlukan beberapa menit)\n')

# Aktifkan progress bar dari tqdm untuk menampilkan persentase progress
tqdm.pandas(desc='Tokenisasi Stanza')

# Aplikasikan fungsi tokenize_stanza ke setiap baris kolom 'judul_lower'
# Hasilnya disimpan di kolom baru 'judul_tokenized'
df['judul_tokenized'] = df['judul_lower'].progress_apply(tokenize_stanza)

print('\nTokenisasi selesai!')
print('\nContoh hasil tokenisasi:')
print('=' * 70)
# Tampilkan perbandingan input (judul_lower) vs output (judul_tokenized)
for _, row in df[['judul_lower', 'judul_tokenized']].head(5).iterrows():
    print(f'INPUT  : {row["judul_lower"]}')
    print(f'OUTPUT : {row["judul_tokenized"]}')
    print('-' * 70)

⚠️  GPU error: cuDNN version 91900 is not compatible with devices with SM < 7.5. Please install...
   Fallback ke CPU...
✓ Stanza initialized on CPU
Memproses tokenisasi Stanza untuk 10,000 judul...
(Proses ini memerlukan beberapa menit)



Tokenisasi Stanza: 100%|██████████| 10000/10000 [01:21<00:00, 122.73it/s]


Tokenisasi selesai!

Contoh hasil tokenisasi:
INPUT  : unboxing iphone 17 pro palsu yang sangat mirip aslinya...
OUTPUT : unboxing iphone 17 pro palsu yang sangat mirip aslinya . . .
----------------------------------------------------------------------
INPUT  : raja terakhir hp samsung!
OUTPUT : raja terakhir hp samsung !
----------------------------------------------------------------------
INPUT  : rp1.599 juta! ketika oppo niat bikin hp murah...
OUTPUT : rp1.599 juta ! ketika oppo niat bikin hp murah . . .
----------------------------------------------------------------------
INPUT  : kalau apple niat, iphone bisa seworth it ini... - review iphone 17
OUTPUT : kalau apple niat , iphone bisa seworth it ini . . . - review iphone 17
----------------------------------------------------------------------
INPUT  : xiaomi pun ngeluh soal fenomena ini...
OUTPUT : xiaomi pun ngeluh soal fenomena ini . . .
----------------------------------------------------------------------


---
## 8. Split Data Terintegrasi 70/10/20 (Train/Validation/Test Channel)

Split utama mengikuti skema **70/10/20** di level channel:
- **Train channels (70%)**: untuk fine-tuning model dan recommendation pool
- **Validation channels (10%)**: khusus validasi fine-tuning
- **Test channels (20%)**: untuk evaluasi sistem rekomendasi

In [ ]:
# Bagi channel ke train/validation/test dan siapkan label untuk fine-tuning
from sklearn.model_selection import train_test_split

# Ambil data unik per channel beserta kategorinya (1 baris = 1 channel)
channel_level_df = (
    df.drop_duplicates(subset='nama_channel')[['nama_channel', 'kategori']]
      .reset_index(drop=True)
)

# Split pertama: 70% train, 30% sisanya (untuk val+test)
# stratify memastikan distribusi kategori sama di setiap split
train_ch_df, temp_ch_df = train_test_split(
    channel_level_df,
    test_size=0.3,
    random_state=42,
    stratify=channel_level_df['kategori']
)

# Split kedua: dari 30% sisanya, bagi 2/3 jadi test (20% total), 1/3 jadi val (10% total)
val_ch_df, test_ch_df = train_test_split(
    temp_ch_df,
    test_size=2/3,  # 20% total test, 10% total validation
    random_state=42,
    stratify=temp_ch_df['kategori']
)

# Konversi DataFrame channel menjadi list nama channel untuk kemudahan filtering
train_channels = train_ch_df['nama_channel'].tolist()
val_channels = val_ch_df['nama_channel'].tolist()
test_channels = test_ch_df['nama_channel'].tolist()

# Fungsi helper untuk membuat dataset yang sudah di-filter dan dibersihkan
def build_supervised_split(channel_list):
    # Filter hanya video dari channel yang ada di channel_list
    out = df[df['nama_channel'].isin(channel_list)][['judul_tokenized', 'kategori']].dropna().copy()
    # Hapus judul yang kosong atau hanya spasi (untuk data berkualitas)
    out = out[out['judul_tokenized'].str.strip() != ''].reset_index(drop=True)
    return out

# Buat 3 dataset terpisah untuk train, validation, dan test
train_sup_df = build_supervised_split(train_channels)
val_sup_df = build_supervised_split(val_channels)
test_sup_df = build_supervised_split(test_channels)

# Gabungkan semua dataset untuk mendapatkan list label unik yang konsisten
supervised_pool = pd.concat([train_sup_df, val_sup_df, test_sup_df], ignore_index=True)

# Buat mapping antara kategori (string) ↔ ID (integer) untuk fine-tuning model
label_list = sorted(supervised_pool['kategori'].unique().tolist())  # Urutkan agar konsisten
label2id = {label: i for i, label in enumerate(label_list)}  # {'Animals': 0, 'Automotive': 1, ...}
id2label = {i: label for label, i in label2id.items()}  # {0: 'Animals', 1: 'Automotive', ...}

# Siapkan dataset final dengan kolom label_id untuk supervised learning
train_df = train_sup_df.copy()
val_df = val_sup_df.copy()
test_df = test_sup_df.copy()

# Mapping kategori ke ID di setiap dataset
train_df['label_id'] = train_df['kategori'].map(label2id)
val_df['label_id'] = val_df['kategori'].map(label2id)
test_df['label_id'] = test_df['kategori'].map(label2id)

# Tampilkan ringkasan split data
print('=' * 85)
print('RINGKASAN SPLIT DATA 70/10/20 (LEVEL CHANNEL)')
print('=' * 85)
print(f'Total channel                    : {len(channel_level_df):,}')
print(f'Train channels (70%)             : {len(train_channels):,}')
print(f'Validation channels (10%)        : {len(val_channels):,}')
print(f'Test channels (20%)              : {len(test_channels):,}')
print('-' * 85)
print(f'Train videos (fine-tuning)       : {len(train_df):,}')
print(f'Validation videos (fine-tuning)  : {len(val_df):,}')
print(f'Test videos (evaluation)         : {len(test_df):,}')
print(f'Jumlah label kategori            : {len(label_list)}')
print('=' * 85)

RINGKASAN SPLIT DATA 70/10/20 (LEVEL CHANNEL)
Total channel                    : 100
Train channels (70%)             : 70
Validation channels (10%)        : 10
Test channels (20%)              : 20
-------------------------------------------------------------------------------------
Train videos (fine-tuning)       : 7,000
Validation videos (fine-tuning)  : 1,000
Test videos (evaluation)         : 2,000
Jumlah label kategori            : 10


---
## 9. Konfigurasi Model IndoBERT

Model dasar yang digunakan untuk fine-tuning adalah `indolem/indobert-base-uncased`.

In [ ]:
# Siapkan nama model IndoBERT sebelum masuk ke tahap fine-tuning
INDOBERT_MODEL_NAME = 'indolem/indobert-base-uncased'

print(f'Model dasar fine-tuning : {INDOBERT_MODEL_NAME}')
print(f'Device                 : {device}')

# Cek tokenizer saja di tahap ini (model akan di-load saat training fine-tuning)
tokenizer_preview = AutoTokenizer.from_pretrained(INDOBERT_MODEL_NAME)
print(f'Vocab size tokenizer   : {tokenizer_preview.vocab_size:,}')

Model dasar fine-tuning : indolem/indobert-base-uncased
Device                 : cpu


Vocab size tokenizer   : 31,923


---
## 10. Fine-Tuning IndoBERT (Supervised Classification)

Fine-tuning dilakukan pada data train dan dipantau menggunakan validation set.
Objective: klasifikasi kategori judul video menggunakan cross-entropy loss.


Fine-tune IndoBERT model untuk klasifikasi kategori judul video.
Parameters:
- train_df: DataFrame dengan kolom 'judul_tokenized' dan 'label_id'
- val_df: DataFrame validation dengan format sama
- model_name: Nama model pre-trained dari HuggingFace
- num_epochs: Berapa kali iterate seluruh training data
- batch_size: Jumlah sample per batch
- lr: Learning rate optimizer
- max_length: Panjang maksimal token

In [ ]:
# Definisikan dataset, data loader, dan loop training fine-tuning IndoBERT
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import copy

# 1. CUSTOM DATASET CLASS - Mengubah data menjadi format yang PyTorch pahami
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts  # List of text strings
        self.labels = labels  # List of label integers

    def __len__(self):
        """Mengembalikan total jumlah sample dalam dataset"""
        return len(self.texts)

    def __getitem__(self, idx):
        """Mengambil satu sample pada index tertentu"""
        return self.texts[idx], self.labels[idx]

# 2. COLLATE FUNCTION - Preprocessing batch data (tokenisasi & padding)
def collate_batch(batch, tokenizer, max_length=128):
    texts, labels = zip(*batch)  # Pisahkan texts dan labels
    
    # Tokenisasi semua teks dalam batch sekaligus (efficient batching)
    enc = tokenizer(
        list(texts),
        padding=True,          # Padding otomatis ke panjang maksimal dalam batch
        truncation=True,       # Potong teks yang lebih panjang dari max_length
        max_length=max_length, # Panjang maksimal token
        return_tensors='pt'    # Return PyTorch tensors (bukan list)
    )
    
    # Convert labels ke PyTorch tensor
    labels = torch.tensor(labels, dtype=torch.long)
    return enc, labels

# 3. EVALUATION FUNCTION - Hitung akurasi di validation/test set
def evaluate_classifier(model, data_loader, device):
    model.eval()  # Set model ke evaluation mode (matikan dropout, etc)
    total = 0
    correct = 0

    with torch.no_grad():  # Tidak perlu compute gradients (hemat memory)
        for batch_inputs, batch_labels in data_loader:
            # Pindahkan data ke GPU/CPU sesuai device
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            batch_labels = batch_labels.to(device)

            # Forward pass: prediksi output
            outputs = model(**batch_inputs)
            
            # Ambil class dengan probabilitas tertinggi (argmax)
            preds = outputs.logits.argmax(dim=-1)

            # Hitung jumlah prediksi yang benar
            total += batch_labels.size(0)
            correct += (preds == batch_labels).sum().item()

    # Return akurasi (correct predictions / total samples)
    return correct / max(total, 1)

# 4. MAIN FINE-TUNING FUNCTION
def train_classifier_fine_tune(train_df, val_df, model_name='indolem/indobert-base-uncased',
                               num_epochs=3, batch_size=16, lr=2e-5, max_length=128):
    
    # Load tokenizer dari model pre-trained
    tokenizer_local = AutoTokenizer.from_pretrained(model_name)
    num_labels = len(label_list)  # Jumlah kategori (10)

    # Load pre-trained IndoBERT untuk sequence classification
    clf_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,      # Mapping: {0: 'Animals', 1: 'Automotive', ...}
        label2id=label2id       # Mapping: {'Animals': 0, 'Automotive': 1, ...}
    )
    clf_model = clf_model.to(device)  # Pindahkan model ke GPU

    # Buat custom Dataset objects
    train_dataset = TextClassificationDataset(
        train_df['judul_tokenized'].tolist(),
        train_df['label_id'].tolist()
    )
    val_dataset = TextClassificationDataset(
        val_df['judul_tokenized'].tolist(),
        val_df['label_id'].tolist()
    )

    # Buat DataLoaders (handle batching, shuffling, preprocessing)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,  # Acak order training data setiap epoch
        collate_fn=lambda b: collate_batch(b, tokenizer_local, max_length=max_length)
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,  # Jangan acak validation data
        collate_fn=lambda b: collate_batch(b, tokenizer_local, max_length=max_length)
    )

    # Setup optimizer & scheduler (mengatur learning rate selama training)
    optimizer = torch.optim.AdamW(clf_model.parameters(), lr=lr)
    
    # Learning rate schedule: warmup awal, gradual decay
    total_steps = max(len(train_loader) * num_epochs, 1)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),      # 10% warmup
        num_training_steps=total_steps
    )

    # Inisialisasi tracking variables
    history = []
    best_val_acc = -1.0
    best_state_dict = copy.deepcopy(clf_model.state_dict())  # Simpan best weights
    best_epoch = 0

    # TRAINING LOOP - iterasi per epoch
    for epoch in range(1, num_epochs + 1):
        clf_model.train()  # Set model ke training mode
        running_loss = 0.0

        # Iterate melalui setiap batch di training set
        for batch_inputs, batch_labels in tqdm(train_loader, desc=f'Fine-tune epoch {epoch}/{num_epochs}', unit='batch'):
            # Pindahkan batch ke GPU
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            batch_labels = batch_labels.to(device)

            # Forward pass: compute loss
            outputs = clf_model(**batch_inputs, labels=batch_labels)
            loss = outputs.loss

            # Backward pass: compute gradients
            optimizer.zero_grad()  # Reset gradients
            loss.backward()        # Backpropagation
            optimizer.step()       # Update weights
            scheduler.step()       # Update learning rate

            running_loss += loss.item()

        # Evaluasi setelah 1 epoch selesai
        avg_train_loss = running_loss / max(len(train_loader), 1)
        val_acc = evaluate_classifier(clf_model, val_loader, device)
        history.append({'epoch': epoch, 'train_loss': round(avg_train_loss, 4), 'val_acc': round(val_acc, 4)})
        print(f'  Epoch {epoch}/{num_epochs} | train_loss={avg_train_loss:.4f} | val_acc={val_acc:.4f}')

        # Simpan model terbaik (berdasarkan validation accuracy tertinggi)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state_dict = copy.deepcopy(clf_model.state_dict())
        
        # EARLY STOPPING - hentikan jika val_acc turun (overfitting detected)
        elif epoch > 1 and val_acc < history[-2]['val_acc']:
            print(f'  Validation accuracy turun dari epoch {epoch-1} ke epoch {epoch}. Early stopping dan pakai model terbaik di epoch {best_epoch}.')
            break

    # Load best model weights sebelum return
    clf_model.load_state_dict(best_state_dict)

    # Return hasil fine-tuning
    return {
        'model': clf_model,
        'tokenizer': tokenizer_local,
        'history': pd.DataFrame(history),
        'best_val_acc': float(best_val_acc if best_val_acc >= 0 else 0.0),
        'best_epoch': best_epoch
    }

# HYPERPARAMETERS - Konfigurasi training
NUM_EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
MAX_LENGTH = 128

print('=' * 80)
print('FINE-TUNING INDOBERT')
print('=' * 80)
print(f'Dataset          : {len(train_df):,} train, {len(val_df):,} val')
print(f'Labels           : {len(label_list)} kategori')
print(f'Epochs           : {NUM_EPOCHS}')
print(f'Batch size       : {BATCH_SIZE}')
print(f'Learning rate    : {LEARNING_RATE}')
print(f'Device           : {device}')
print('=' * 80)

# JALANKAN FINE-TUNING
fine_tuned_result = train_classifier_fine_tune(
    train_df=train_df,
    val_df=val_df,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    max_length=MAX_LENGTH
)

print('\nFine-tuning selesai.')
print(f'Best validation accuracy: {fine_tuned_result["best_val_acc"]:.4f}')
print(f'Best epoch             : {fine_tuned_result["best_epoch"]}')

FINE-TUNING INDOBERT
Dataset          : 7,000 train, 1,000 val
Labels           : 10 kategori
Epochs           : 10
Batch size       : 16
Learning rate    : 2e-05
Device           : cpu


[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9518.49it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not 

  Epoch 1/10 | train_loss=1.9374 | val_acc=0.6650


Fine-tune epoch 2/10: 100%|██████████| 438/438 [21:18<00:00,  2.92s/batch]


  Epoch 2/10 | train_loss=0.6858 | val_acc=0.7280


...

  Epoch 4/10 | train_loss=0.2209 | val_acc=0.7710


Fine-tune epoch 5/10: 100%|██████████| 438/438 [20:38<00:00,  2.83s/batch]


  Epoch 5/10 | train_loss=0.1304 | val_acc=0.7840


Fine-tune epoch 6/10: 100%|██████████| 438/438 [20:41<00:00,  2.83s/batch]


  Epoch 6/10 | train_loss=0.0792 | val_acc=0.7810
  Validation accuracy turun dari epoch 5 ke epoch 6. Early stopping dan pakai model terbaik di epoch 5.

Fine-tuning selesai.
Best validation accuracy: 0.7840
Best epoch             : 5


---
## 11. Generate Embedding Judul Video (Fine-Tuned Encoder)

Embedding dibuat menggunakan encoder hasil fine-tuning dengan metode masked mean pooling.

In [ ]:
# Ubah judul video menjadi embedding dengan encoder hasil fine-tuning
def generate_finetuned_embeddings(texts, tokenizer_local, encoder_model, device, batch_size=32, max_length=128):
    # Mode evaluasi: mematikan dropout agar hasil embedding konsisten
    encoder_model.eval()
    all_emb = []

    # Proses data per batch supaya lebih hemat memori dan lebih cepat
    for i in tqdm(range(0, len(texts), batch_size), desc='Generating fine-tuned embeddings', unit='batch'):
        batch_texts = texts[i:i + batch_size]

        # Tokenisasi teks: padding agar panjang input dalam batch sama
        inputs = tokenizer_local(
            batch_texts,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=max_length
        )

        # Pindahkan tensor ke device yang dipakai (GPU/CPU)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Ambil output hidden state dari encoder IndoBERT
        with torch.no_grad():
            out = encoder_model(**inputs).last_hidden_state

        # Mask dipakai agar token padding tidak ikut dihitung saat pooling
        mask = inputs['attention_mask'].unsqueeze(-1).float()

        # Mean pooling: jumlahkan embedding token aktif lalu dibagi jumlah token aktif
        summed = (out * mask).sum(dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        pooled = summed / counts

        # Simpan embedding batch ke list
        all_emb.append(pooled.cpu().numpy())

    # Gabungkan semua batch menjadi satu array embedding
    return np.vstack(all_emb)

# Ambil seluruh judul yang sudah ditokenisasi
texts_to_embed = df['judul_tokenized'].tolist()

# Ambil encoder hasil fine-tuning dari model klasifikasi
ft_encoder = fine_tuned_result['model'].base_model

# Ambil tokenizer yang sama dengan saat fine-tuning
ft_tokenizer = fine_tuned_result['tokenizer']

print('=' * 80)
print('GENERATE EMBEDDING DENGAN MODEL FINE-TUNED')
print('=' * 80)

# Jalankan pembuatan embedding untuk semua video
ft_video_embeddings = generate_finetuned_embeddings(
    texts=texts_to_embed,
    tokenizer_local=ft_tokenizer,
    encoder_model=ft_encoder,
    device=device,
    batch_size=32,
    max_length=MAX_LENGTH
)

# Buat salinan dataframe asli lalu tambahkan kolom embedding
df_ft = df.copy()
df_ft['embedding'] = list(ft_video_embeddings)

# Cek hasil embedding
print(f'Jumlah embedding: {ft_video_embeddings.shape[0]:,}')
print(f'Dimensi vektor  : {ft_video_embeddings.shape[1]}')

GENERATE EMBEDDING DENGAN MODEL FINE-TUNED


Generating fine-tuned embeddings: 100%|██████████| 313/313 [06:54<00:00,  1.32s/batch]

Jumlah embedding: 10,000
Dimensi vektor  : 768


---
## 12. Agregasi Vektor Channel (Mean Pooling + L2 Normalization)

Setiap channel direpresentasikan sebagai rata-rata embedding seluruh videonya.

Cara kerja:
1. Kelompokkan semua video berdasarkan nama_channel
2. Untuk setiap channel, hitung rata-rata (mean) dari semua embedding videonya
3. Hasilnya adalah 1 embedding per channel (bukan per video)
    
Parameter:
- input_df: DataFrame yang memiliki kolom 'nama_channel' dan 'embedding'
    
Return:
- Dictionary dengan key=nama_channel, value=embedding rata-rata (numpy array)

In [ ]:
# Agregasikan embedding per channel lalu normalisasi agar siap dihitung similarity
from sklearn.preprocessing import normalize

def aggregate_channel_embeddings(input_df):
    # Fungsi untuk mengagregasi (merata-ratakan) embedding video menjadi embedding per channel.
    grouped = input_df.groupby('nama_channel')
    return {
        ch: np.mean(np.stack(group['embedding'].values), axis=0)
        for ch, group in tqdm(grouped, desc='Agregasi channel', unit='channel')
    }

# Jalankan fungsi agregasi untuk mendapatkan embedding per channel
# ft_video_embeddings sudah ada dari cell sebelumnya (fine-tuned embeddings)
channel_vectors_ft = aggregate_channel_embeddings(df_ft)

# Ambil nama-nama channel dan urutkan agar konsisten (penting untuk matrix)
channel_names_ft = list(channel_vectors_ft.keys())

# Buat matrix dengan stack: setiap baris = 1 channel, setiap kolom = 1 dimensi embedding
# Hasil: shape (jumlah_channel, 768) karena IndoBERT base punya 768 dimensi
ft_channel_matrix = np.stack([channel_vectors_ft[ch] for ch in channel_names_ft])

# L2 Normalization: agar setiap vektor channel memiliki norm 1
# Ini penting untuk cosine similarity agar hasilnya lebih akurat dan stabil
# Rumus: vektor_baru = vektor_lama / ||vektor_lama||
ft_channel_matrix = normalize(ft_channel_matrix, norm='l2')

print('=' * 80)
print('AGREGASI CHANNEL FINE-TUNED')
print('=' * 80)
print(f'Jumlah channel : {len(channel_names_ft)}')
print(f'Shape matrix   : {ft_channel_matrix.shape}')
print('(Setiap baris = 1 channel, Setiap kolom = 1 dimensi embedding)')

Agregasi channel: 100%|██████████| 100/100 [00:00<00:00, 1375.01channel/s]

AGREGASI CHANNEL FINE-TUNED
Jumlah channel : 100
Shape matrix   : (100, 768)


---
## 13. Cosine Similarity Matrix & Fungsi Rekomendasi

Menghitung similarity antar channel dengan recommendation pool dari train channels dan menampilkan rekomendasi Top-K.

In [ ]:
# Hitung cosine similarity antar channel berdasarkan vektor embedding
ft_similarity_matrix = cosine_similarity(ft_channel_matrix)

# Ubah matrix similarity menjadi DataFrame agar lebih mudah diakses per nama channel
ft_similarity_df = pd.DataFrame(
    ft_similarity_matrix,
    index=channel_names_ft,
    columns=channel_names_ft
)

def recommend_channel(channel_name, top_k, similarity_df, channel_info, recommendation_pool):
    # Ambil skor kemiripan untuk channel input
    # lalu buang dirinya sendiri agar tidak merekomendasikan channel yang sama
    sim_scores = similarity_df.loc[channel_name].drop(labels=channel_name)

    # Batasi kandidat hanya dari recommendation pool (misalnya train channels)
    valid_candidates = [ch for ch in recommendation_pool if ch in sim_scores.index]
    sim_scores = sim_scores.loc[valid_candidates]

    # Ambil top-K channel dengan similarity tertinggi
    top_k_channels = sim_scores.sort_values(ascending=False).head(top_k)

    rows = []
    for rank, (ch, score) in enumerate(top_k_channels.items(), start=1):
        # Ambil informasi tambahan channel, jika tidak ada isi default
        info = channel_info.loc[ch] if ch in channel_info.index else {'kategori': '-', 'jumlah_pelanggan': 0}

        # Simpan hasil rekomendasi ke list
        rows.append({
            'rank': rank,
            'nama_channel': ch,
            'kategori': info['kategori'],
            'jumlah_pelanggan': info['jumlah_pelanggan'],
            'similarity_score': round(float(score), 4)
        })

    # Jadikan DataFrame agar hasil mudah dibaca
    return pd.DataFrame(rows).set_index('rank')

# Ambil info channel unik untuk kebutuhan menampilkan kategori dan jumlah pelanggan
channel_info = (
    df.drop_duplicates(subset='nama_channel')
      .set_index('nama_channel')[['kategori', 'jumlah_pelanggan']]
)

# Contoh input channel dari test set
input_channel = test_channels[0]
TOP_K = 5

# Jalankan fungsi rekomendasi
rec_df = recommend_channel(
    channel_name=input_channel,
    top_k=TOP_K,
    similarity_df=ft_similarity_df,
    channel_info=channel_info,
    recommendation_pool=train_channels
)

print('=' * 80)
print('CONTOH REKOMENDASI (FINE-TUNED)')
print('=' * 80)
print(f'Input channel (dari test set): {input_channel}')
print(f'Recommendation pool (dari train set): {len(train_channels)} channels')
print(rec_df.to_string())

CONTOH REKOMENDASI (FINE-TUNED)
Input channel (dari test set): @metrotvnews
Recommendation pool (dari train set): 70 channels
            nama_channel kategori  jumlah_pelanggan  similarity_score
rank                                                                 
1              @kompastv     News          19700000            0.9051
2         @CNNIDOFFICIAL     News          11900000            0.9050
3        @NarasiNewsroom     News           1660000            0.9042
4     @tempovideochannel     News           1880000            0.8996
5         @liputan6_news     News           2670000            0.8982


---
## 14. Evaluasi Sistem (Precision@K pada Test Channels)

Evaluasi dilakukan pada **test channels (20%)** dengan definisi relevansi kategori yang sama.
Recommendation pool hanya berasal dari **train channels (70%)**.

In [ ]:
# Evaluasi Precision@K pada test channels untuk mengukur kualitas rekomendasi
# Precision@K = proporsi top-K rekomendasi yang relevan (kategori sama) dengan channel target
def precision_at_k(channel_name, k, similarity_df, channel_category_map, recommendation_pool):
    # Ambil skor similarity untuk channel input
    # Channel itu sendiri dihapus supaya tidak merekomendasikan dirinya sendiri
    sim_scores = similarity_df.loc[channel_name].drop(labels=channel_name)

    # Batasi kandidat rekomendasi hanya pada channel yang ada di recommendation pool
    valid_candidates = [ch for ch in recommendation_pool if ch in sim_scores.index]
    sim_scores = sim_scores.loc[valid_candidates]

    # Ambil top-K channel dengan skor similarity tertinggi
    top_k_channels = sim_scores.sort_values(ascending=False).head(k).index.tolist()

    # Kategori asli dari channel target
    target_category = channel_category_map.get(channel_name)

    # Hitung berapa rekomendasi top-K yang kategorinya sama dengan target
    relevant_count = sum(channel_category_map.get(ch) == target_category for ch in top_k_channels)

    # Precision@K = jumlah relevan dibagi K
    return relevant_count / k


def evaluate_system(k_values, eval_channels, similarity_df, channel_category_map, recommendation_pool):
    # Menyimpan hasil evaluasi per channel
    rows = []

    # Loop semua channel test
    for ch in tqdm(eval_channels, desc='Evaluasi Precision@K', unit='channel'):
        # Simpan nama channel dan kategorinya
        row = {'channel': ch, 'kategori': channel_category_map.get(ch, '-')}

        # Hitung Precision@K untuk setiap nilai K
        for k in k_values:
            row[f'P@{k}'] = round(
                precision_at_k(ch, k, similarity_df, channel_category_map, recommendation_pool),
                4
            )

        rows.append(row)

    # Ubah hasil menjadi DataFrame dan jadikan nama channel sebagai index
    eval_out = pd.DataFrame(rows).set_index('channel')

    # Hitung rata-rata Precision@K dari semua channel test
    avg_row = {'kategori': 'AVERAGE'}
    for k in k_values:
        avg_row[f'P@{k}'] = round(eval_out[f'P@{k}'].mean(), 4)

    # Tambahkan baris average di bawah hasil evaluasi
    return pd.concat([eval_out, pd.DataFrame([avg_row], index=['--- AVERAGE ---'])])


# Mapping nama channel -> kategori
# Dipakai untuk mengecek apakah rekomendasi berada di kategori yang sama
channel_category_map = (
    df.drop_duplicates(subset='nama_channel')
      .set_index('nama_channel')['kategori']
      .to_dict()
)

# Nilai K yang akan diuji pada evaluasi Precision@K
K_VALUES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Jalankan evaluasi pada semua test channels
# Recommendation pool hanya dari train channels agar evaluasi lebih realistis
ft_eval_df = evaluate_system(
    k_values=K_VALUES,
    eval_channels=test_channels,
    similarity_df=ft_similarity_df,
    channel_category_map=channel_category_map,
    recommendation_pool=train_channels
)

# Tampilkan hasil evaluasi
print('=' * 80)
print('HASIL EVALUASI FINE-TUNED (Precision@K - Test Channels)')
print('=' * 80)
print(ft_eval_df.to_string())

Evaluasi Precision@K:   0%|          | 0/20 [00:00<?, ?channel/s]

Evaluasi Precision@K: 100%|██████████| 20/20 [00:00<00:00, 102.13channel/s]


HASIL EVALUASI FINE-TUNED (Precision@K - Test Channels)
                         kategori  P@1  P@2  P@3  P@4  P@5  P@6  P@7    P@8     P@9  P@10
@metrotvnews                 News  1.0  1.0  1.0  1.0  1.0  1.0  1.0  0.875  0.7778  0.70
@Audrey-A                 Animals  1.0  1.0  1.0  1.0  1.0  1.0  1.0  0.875  0.7778  0.70
@BimbelBrilian          Education  1.0  1.0  1.0  1.0  1.0  1.0  1.0  0.875  0.7778  0.70
@LuckyHakimChannel        Animals  1.0  1.0  1.0  1.0  1.0  1.0  1.0  0.875  0.7778  0.70
@GadgetIn                 Gadgets  1.0  1.0  1.0  1.0  1.0  1.0  1.0  0.875  0.7778  0.70
@AttaHalilintar     Entertainment  1.0  1.0  1.0  1.0  1.0  1.0  1.0  0.875  0.7778  0.70
@ZeniusEducation        Education  1.0  1.0  1.0  1.0  1.0  1.0  1.0  0.875  0.7778  0.70
@AfifYulistian             Gaming  1.0  1.0  1.0  1.0  1.0  1.0  1.0  0.875  0.7778  0.70
@Anak.Kuliner                Food  1.0  1.0  1.0  1.0  1.0  1.0  1.0  0.875  0.7778  0.70
@GarasiDrift           Automotive  1.0  1.0 

In [ ]:
# Ringkas hasil Precision@K per kategori agar lebih mudah dibaca
print('\nRingkasan Rata-Rata Precision@K untuk Test Channels:')
avg_series = ft_eval_df.loc['--- AVERAGE ---']
for k in K_VALUES:
    print(f'  Precision@{k:>2} = {avg_series[f"P@{k}"]:.4f}')

# Ambil hanya baris channel (hapus baris AVERAGE) untuk analisis per kategori
# Jadi kita fokus ke data channel individu, bukan rata-rata keseluruhan
ft_eval_df_clean = ft_eval_df[ft_eval_df['kategori'] != 'AVERAGE'].copy()

# Kumpulkan nama kolom Precision@K (P@1, P@2, ..., P@10) ke dalam list
# Ini dipakai untuk groupby dan mean calculation di langkah berikutnya
pk_cols = [f'P@{k}' for k in K_VALUES]

# Kelompokkan channel berdasarkan kategori, lalu hitung rata-rata Precision@K per kategori
# Contoh: semua channel di kategori 'Animals' dirata-ratakan Precision@1-nya
cat_summary = (
    ft_eval_df_clean.groupby('kategori')[pk_cols]
    .mean()
    .round(4)
)

# Hitung rata-rata semua Precision@K (P@1 sampai P@10) untuk setiap kategori
# Ini adalah skor keseluruhan kemiripan per kategori (semakin tinggi semakin baik)
cat_summary['avg_precision'] = cat_summary[pk_cols].mean(axis=1)

# Urutkan kategori berdasarkan avg_precision dari tertinggi ke terendah
# Kategori dengan skor tinggi berarti rekomendasi sistemnya sangat akurat untuk kategori itu
cat_summary = cat_summary.sort_values('avg_precision', ascending=False)

# Ambil 10 kategori teratas (terbaik) berdasarkan avg_precision
# Index ini akan dipakai untuk menampilkan top-10 kategori di output
top10_cat_indices = cat_summary.head(10).index

print('\nTop 10 Kategori dengan Kemiripan Tertinggi (Precision@1 - Precision@10):')
print('=' * 120)
# Loop setiap kategori di top-10 dan tampilkan precision score-nya untuk setiap K
for i, kategori in enumerate(top10_cat_indices, start=1):
    # Buat string berisi semua precision score dari P@1 sampai P@10 untuk kategori tersebut
    precision_str = ' | '.join([f'P@{k}={cat_summary.loc[kategori, f"P@{k}"]:.4f}' for k in K_VALUES])
    # Cetak urutan, nama kategori, dan semua precision score-nya
    print(f'{i:>2}. {kategori:<25} | {precision_str}')
print('=' * 120)


Ringkasan Rata-Rata Precision@K untuk Test Channels:
  Precision@ 1 = 0.9000
  Precision@ 2 = 0.9000
  Precision@ 3 = 0.9000
  Precision@ 4 = 0.9000
  Precision@ 5 = 0.9000
  Precision@ 6 = 0.9000
  Precision@ 7 = 0.9000
  Precision@ 8 = 0.8000
  Precision@ 9 = 0.7222
  Precision@10 = 0.6600

Top 10 Kategori dengan Kemiripan Tertinggi (Precision@1 - Precision@10):
 1. Animals                   | P@1=1.0000 | P@2=1.0000 | P@3=1.0000 | P@4=1.0000 | P@5=1.0000 | P@6=1.0000 | P@7=1.0000 | P@8=0.8750 | P@9=0.7778 | P@10=0.7000
 2. Automotive                | P@1=1.0000 | P@2=1.0000 | P@3=1.0000 | P@4=1.0000 | P@5=1.0000 | P@6=1.0000 | P@7=1.0000 | P@8=0.8750 | P@9=0.7778 | P@10=0.7000
 3. Education                 | P@1=1.0000 | P@2=1.0000 | P@3=1.0000 | P@4=1.0000 | P@5=1.0000 | P@6=1.0000 | P@7=1.0000 | P@8=0.8750 | P@9=0.7778 | P@10=0.7000
 4. Food                      | P@1=1.0000 | P@2=1.0000 | P@3=1.0000 | P@4=1.0000 | P@5=1.0000 | P@6=1.0000 | P@7=1.0000 | P@8=0.8750 | P@9=0.7778 | 

---
## 15. Simpan Artefak Final Fine-Tuning

Semua artefak final disimpan ke folder `output/fine_tuning_experiment/`.

In [ ]:
# Simpan semua artefak final agar notebook bisa dipakai ulang tanpa rerun penuh
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

FT_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'fine_tuning_experiment')
os.makedirs(FT_OUTPUT_DIR, exist_ok=True)

print('=' * 80)
print(f'Menyimpan artefak fine-tuning ke: {FT_OUTPUT_DIR}')
print('=' * 80)

# 1) Split fine-tuning
train_df.to_csv(os.path.join(FT_OUTPUT_DIR, 'train_split.csv'), index=False)
val_df.to_csv(os.path.join(FT_OUTPUT_DIR, 'val_split.csv'), index=False)

# 2) Riwayat training
fine_tuned_result['history'].to_csv(os.path.join(FT_OUTPUT_DIR, 'training_history.csv'), index=False)

# 3) Hasil evaluasi rekomendasi
ft_eval_df.to_csv(os.path.join(FT_OUTPUT_DIR, 'evaluation_precision_at_k.csv'))

# 4) Matrix hasil fine-tuned
ft_similarity_df.to_csv(os.path.join(FT_OUTPUT_DIR, 'similarity_matrix_fine_tuned.csv'))
np.save(os.path.join(FT_OUTPUT_DIR, 'channel_matrix_fine_tuned.npy'), ft_channel_matrix)
np.save(os.path.join(FT_OUTPUT_DIR, 'video_embeddings_fine_tuned.npy'), ft_video_embeddings)

# 5) Model fine-tuned
fine_tuned_result['model'].save_pretrained(os.path.join(FT_OUTPUT_DIR, 'model_fine_tuned'))
fine_tuned_result['tokenizer'].save_pretrained(os.path.join(FT_OUTPUT_DIR, 'model_fine_tuned'))

# 6) Channel names (ordered to match channel matrix) - saved at top-level output/
channel_names_df = pd.DataFrame({'nama_channel': channel_names_ft})
channel_names_df.to_csv(os.path.join(OUTPUT_DIR, 'channel_names.csv'), index=False)

# 7) Video metadata (one row per channel) - saved at top-level output/
video_metadata_cols = ['nama_channel', 'kategori', 'jumlah_pelanggan', 'link_channel']
if 'nama_channel' in df.columns:
    video_metadata_df = df.drop_duplicates(subset='nama_channel').copy()
else:
    video_metadata_df = pd.DataFrame({'nama_channel': channel_names_ft})

# Ensure expected columns present
for col in video_metadata_cols:
    if col not in video_metadata_df.columns:
        video_metadata_df[col] = '-' if col in ['kategori', 'link_channel'] else 0

video_metadata_df = video_metadata_df[video_metadata_cols]
video_metadata_df.to_csv(os.path.join(OUTPUT_DIR, 'video_metadata.csv'), index=False)

# 8) Metadata
metadata = {
    'model_name': INDOBERT_MODEL_NAME,
    'best_validation_accuracy': float(fine_tuned_result['best_val_acc']),
    'num_epochs': NUM_EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'max_length': MAX_LENGTH,
    'k_values': K_VALUES,
    'train_channels_count': len(train_channels),
    'validation_channels_count': len(val_channels),
    'test_channels_count': len(test_channels),
    'num_labels': len(label_list),
    'labels': label_list
}
with open(os.path.join(FT_OUTPUT_DIR, 'metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('\nArtefak tersimpan:')
for fname in sorted(os.listdir(FT_OUTPUT_DIR)):
    fpath = os.path.join(FT_OUTPUT_DIR, fname)
    print(f'  - {fname}/' if os.path.isdir(fpath) else f'  - {fname}')

print('\nJuga disimpan (top-level output/):')
for fname in ['channel_names.csv', 'video_metadata.csv']:
    print(f'  - {fname}')

print('\nSelesai menyimpan artefak fine-tuning.')

Menyimpan artefak fine-tuning ke: output/fine_tuning_experiment


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.90it/s]


Artefak tersimpan:
  - channel_matrix_fine_tuned.npy
  - evaluation_precision_at_k.csv
  - metadata.json
  - model_fine_tuned/
  - similarity_matrix_fine_tuned.csv
  - train_split.csv
  - training_history.csv
  - val_split.csv
  - video_embeddings_fine_tuned.npy

Juga disimpan (top-level output/):
  - channel_names.csv
  - video_metadata.csv

Selesai menyimpan artefak fine-tuning.


---
## Ringkasan Metodologi

| No | Tahap | Metode | Catatan |
|---|---|---|---|
| 1 | Install & Import Library | Setup dependency dan import modul utama | `torch`, `transformers`, `stanza`, `sklearn`, `pandas`, `numpy`, `tqdm` |
| 2 | Load Dataset JSON | Membaca data dan membentuk DataFrame | `json`, `pandas` |
| 3 | Pengecekan Missing Value | Validasi kualitas data dasar | Cek kolom dan baris yang kosong |
| 4 | Pengecekan Duplikat | Validasi kualitas data dasar | Cek baris video yang sama persis |
| 5 | Text Cleaning | Hapus emoji, simbol non-standar, encoding rusak | `re` |
| 6 | Case Folding | Lowercase seluruh teks | built-in |
| 7 | Tokenisasi dengan Stanza | Tokenisasi Bahasa Indonesia | `stanza` |
| 8 | Split Data Utama | Train/Validation/Test = 70/10/20 (level channel) | Train pool untuk rekomendasi |
| 9 | Konfigurasi Model | Siapkan model dasar IndoBERT | `indolem/indobert-base-uncased` |
| 10 | Fine-Tuning | Klasifikasi kategori judul video | cross-entropy, AdamW, warmup scheduler |
| 11 | Embedding Fine-Tuned | Masked mean pooling dari encoder hasil fine-tuning | `torch`, `numpy` |
| 12 | Agregasi Vektor Channel | Mean pooling antar video per channel + L2 normalization | `numpy`, `sklearn` |
| 13 | Similarity & Rekomendasi | Cosine similarity dan Top-K rekomendasi | `pandas`, `numpy` |
| 14 | Evaluasi | Precision@K pada Test Channels | ground truth: kategori sama |
| 15 | Simpan Artefak | Model, matrix, dan hasil evaluasi final | `os`, `json`, `pandas`, `numpy` |

**Pendekatan:** Content-Based Filtering dengan Fine-Tuned Embedding  
**Model NLP:** `indolem/indobert-base-uncased` (fine-tuned pada train set)  
**Fitur Utama:** Judul Video YouTube